## Example 1: Check whether a string is a syntactically valid IRI

In [ ]:
from rfc3987_syntax import is_valid_syntax_iri

# Example 1: IRIs may carry non-ASCII characters; a space, or a missing scheme, makes the string invalid
valid = [
    "https://example.org/path?query=1#fragment",
    "https://例え.テスト/パス?値=1",
    "mailto:someone@example.org",
    "urn:isbn:0451450523",
]
invalid = ["http://exa mple.org/", "no scheme here", "https://example.org/<bad>"]
for text in valid:
    assert is_valid_syntax_iri(text), text
for text in invalid:
    assert not is_valid_syntax_iri(text), text
print(f"Example 1 OK: {len(valid)} valid IRIs accepted, {len(invalid)} invalid ones refused")

## Example 2: IRI references versus absolute IRIs

In [ ]:
from rfc3987_syntax import is_valid_syntax, is_valid_syntax_iri_reference

# Example 2: a relative reference is a valid "iri_reference" but not an "iri" (which needs a scheme)
relative = "../docs/index.html?lang=fr#top"
assert is_valid_syntax_iri_reference(relative)
assert is_valid_syntax("iri_reference", relative)
assert not is_valid_syntax("iri", relative)

# An absolute IRI has a scheme and no fragment
assert is_valid_syntax("absolute_iri", "https://example.org/a?b=c")
assert not is_valid_syntax("absolute_iri", "https://example.org/a#frag")
print("Example 2 OK: iri, iri_reference and absolute_iri tell the three forms apart")

## Example 3: Parse an IRI into its grammar tree

In [ ]:
from lark import Tree

from rfc3987_syntax import parse

# Example 3: parse() returns a lark parse tree whose nodes are named after the RFC 3987 grammar rules
tree = parse("iri", "https://user@例え.jp:8443/a/b?x=1#frag")
assert isinstance(tree, Tree) and tree.data == "iri", tree.data
rules = {subtree.data for subtree in tree.iter_subtrees()}
for rule in ("scheme", "iauthority", "iquery", "ifragment"):
    assert rule in rules, (rule, sorted(rules))
scheme = next(tree.find_data("scheme"))
scheme_text = "".join(str(token) for token in scheme.scan_values(lambda value: True))
assert scheme_text == "https", scheme_text
print(f"Example 3 OK: {len(rules)} grammar rules in the tree; scheme = {scheme_text!r}")

## Example 4: A validator for any rule of the grammar

In [ ]:
from rfc3987_syntax import RFC3987_SYNTAX_TERMS, make_syntax_validator

# Example 4: make_syntax_validator builds a checker for one grammar rule, here the host part of an IRI
assert "ihost" in RFC3987_SYNTAX_TERMS and "iri" in RFC3987_SYNTAX_TERMS
is_host = make_syntax_validator("ihost")
for host in ("example.org", "例え.テスト", "192.168.0.1", "localhost"):
    assert is_host(host), host
for not_host in ("exa mple.org", "a/b"):
    assert not is_host(not_host), not_host
is_port = make_syntax_validator("port")
assert is_port("8443") and not is_port("84a3")
print("Example 4 OK: host and port validators built from the grammar")

## Example 5: Use it as a JSON Schema format checker

In [ ]:
import jsonschema

from rfc3987_syntax import is_valid_syntax_iri

# Example 5: register the IRI check under a format name, then validate documents against a schema
checker = jsonschema.FormatChecker()
checker.checks("iri-syntax")(is_valid_syntax_iri)
schema = {"type": "object", "properties": {"homepage": {"type": "string", "format": "iri-syntax"}}}
validator = jsonschema.Draft202012Validator(schema, format_checker=checker)
validator.validate({"homepage": "https://例え.テスト/"})
errors = list(validator.iter_errors({"homepage": "not an iri"}))
assert len(errors) == 1 and "iri-syntax" in errors[0].message, [error.message for error in errors]
print("Example 5 OK: jsonschema accepted a valid IRI and reported:", errors[0].message)